
# Gemini Enterprise Agent Platform — Gen AI Hands-On Lab
### GCP Training Program — Day 4-5, Module 6: Gen AI Capabilities

**Problem statement:** we're a training team building intuition for the **Models** side of
Gemini Enterprise Agent Platform (the current name for what was Vertex AI). Using a handful of
tiny, easy-to-read example texts and documents (no real dataset needed — the point is to see
each capability in isolation), this notebook demonstrates:

- **Gemini models** — single-turn, multi-turn chat, multimodal input, function calling, and every
  generation-control knob (`temperature`, `top_p`, `top_k`, etc.)
- **Comparing 2-3 different Gemini models** side by side on the same prompt
- **Context windows** — how much text a model can hold at once, and what changes at scale
- **Embeddings & task types** — the building block behind search and RAG
- **Grounding** — with Google Search (the public web) and with Vector Search (your own documents)
- **Model Garden** and **Agent Studio** (formerly Agent Builder) — guided console walkthroughs

A companion notebook, **Agentic AI on Gemini Enterprise Agent Platform**, picks up where Section 8
leaves off — building an actual agent with the Agent Development Kit (ADK), deploying it to Agent
Runtime, and covering Memory Bank, Agent Registry, Gateway, Policies, Security, and Evaluation.

## ⚠️ Terminology note
As of mid-2026, Google folded **Vertex AI** into the **Gemini Enterprise Agent Platform**. The
console no longer has a "Vertex AI" entry — everything below lives under **Agent Platform > Models**
(what this notebook builds) or **Agent Platform > Agents** (what the companion notebook builds).
The **Python SDK package names are unchanged** — `google-genai` and `google.cloud.aiplatform` still
use those exact names in code, even though the console UI says "Agent Platform."

## ⚠️ SDK note — read this first
This notebook uses the **`google-genai`** SDK (`from google import genai`), Google's current unified
client for Gemini on both Agent Platform and the Gemini Developer API. The older
`vertexai.generative_models` module was **deprecated June 24, 2025 and fully removed June 24, 2026**
— it will throw `NotFound`/`ImportError` errors on any current install, not just a warning. If
anything here stops matching current docs, check:
https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk

Vector Search (Section 6) is unaffected — it lives in `google.cloud.aiplatform`, a separate,
non-deprecated module.

## ⚠️ Cost & trial-account notes
Gemini API calls are fast and cheap (fractions of a cent per call). The one resource that bills
**continuously** here is the **Vector Search index endpoint** in Section 6 — undeploy/delete it
right after the demo (Section 6.4), not at the end of the day. Section 9 does a final cleanup pass
as a safety net.

**This notebook is fully self-contained.** Authenticate → Configure → everything else runs against
your own project.


## 1. Authenticate This Session

In [ ]:

import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import auth
    auth.authenticate_user()
    print("Authenticated in Colab. This also configures gcloud for this session.")
else:
    print("Not running in Colab — assuming gcloud Application Default Credentials are set up.")
    print("If this is your first time, run in a terminal:")
    print("  gcloud auth application-default login")



## 2. Install & Import
`google-genai` is a single, small package — replaces the generative-AI portions of
`google-cloud-aiplatform` entirely. `google-cloud-aiplatform` itself is still needed for Section 6
(Vector Search), which is a separate, non-generative-AI module.


In [ ]:
!pip install -U -q "google-genai" "google-cloud-aiplatform"

In [ ]:

import time
import os
from google import genai
from google.genai.types import (
    Part,
    Tool,
    GoogleSearch,
    FunctionDeclaration,
    GenerateContentConfig,
    EmbedContentConfig,
)

print("google-genai imported successfully")


## 3. Configure Your Project & Region

In [ ]:

PROJECT_ID = input("Enter your GCP Project ID: ").strip()

REGION = input("Enter your Agent Platform region [default: us-central1]: ").strip()
if not REGION:
    REGION = "us-central1"

_suffix = int(time.time())
os.environ["PROJECT_ID"] = PROJECT_ID
os.environ["REGION"] = REGION

!gcloud config set project {PROJECT_ID}

client = genai.Client(vertexai=True, project=PROJECT_ID, location=REGION)

# Centralized model names — defined once, used everywhere below. If any throws a NotFound error,
# confirm current model IDs at https://cloud.google.com/vertex-ai/generative-ai/docs/models and
# update the lines below rather than hunting for hardcoded strings elsewhere in this notebook.
GEMINI_MODEL_NAME = "gemini-2.5-flash"
EMBEDDING_MODEL_NAME = "gemini-embedding-001"

# Used only in Section 4.6 (multi-model comparison) — a lighter and a heavier model alongside
# the default above, so the class sees 3 distinct points on the speed/capability spectrum.
COMPARISON_MODELS = ["gemini-2.5-flash-lite", "gemini-2.5-flash", "gemini-2.5-pro"]

print("Gen AI client initialized.")
print("Default Gemini model:", GEMINI_MODEL_NAME)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Models used in the comparison (Section 4.6):", COMPARISON_MODELS)



## 4. Gemini Models on Agent Platform
### 4.1 Single-Turn Text Generation
**What this does:** the simplest possible Gemini call — one prompt in, one response out, no
conversation history maintained. `client.models.generate_content()` is the single entry point the
Gen AI SDK uses for every non-chat generation call.


In [ ]:

response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents="Explain what a data lake is, in two sentences, for a business audience.",
)
print(response.text)



### 4.2 Multi-Turn Chat
**What this does:** `client.chats.create()` returns a chat session that keeps conversation history
automatically — each `send_message()` call includes everything said before it, so the second
question below can reference "it" and the model resolves that to the topic from the first turn.
This is the simplest possible illustration of **context**: the model's ability to use what came
earlier in the same conversation.


In [ ]:

chat = client.chats.create(model=GEMINI_MODEL_NAME)

r1 = chat.send_message("What is BigQuery partitioning?")
print("Turn 1:", r1.text[:300])

r2 = chat.send_message("Now explain how it differs from clustering, referencing what you just said.")
print("\nTurn 2:", r2.text[:400])



### 4.3 Multimodal Input (Text + Image)
**What this does:** Gemini natively accepts image input alongside text in the same call —
`Part.from_uri` references an image directly from a public GCS URI without needing to download it
first.


In [ ]:

image_part = Part.from_uri(
    file_uri="gs://cloud-samples-data/generative-ai/image/scones.jpg",
    mime_type="image/jpeg",
)

response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=[image_part, "What is in this image? Answer in one sentence."],
)
print(response.text)



### 4.4 Function Calling (Tool Use) — a "Poor Man's Agent"
**What this does:** rather than the model just producing text, we give it a *tool definition* (a
function signature with no actual implementation shown to the model — a bare `FunctionDeclaration`,
not a real callable). When the prompt needs that capability, Gemini returns a structured
function-call request instead of prose — your code would execute the real function and feed the
result back in. This is the same underlying mechanism full agent frameworks (ADK, covered in the
companion notebook) are built on.

**Note:** the Gen AI SDK also supports *automatic* function calling — passing a real Python function
directly as a tool, which the SDK calls and resolves for you without any manual round trip. We use
the manual, schema-only form here specifically so the class can see the raw function-call structure
Gemini returns, rather than have the SDK hide that step.


In [ ]:

get_station_trip_count = FunctionDeclaration(
    name="get_station_trip_count",
    description="Get the number of bike trips that started at a given station name.",
    parameters={
        "type": "object",
        "properties": {
            "station_name": {"type": "string", "description": "The bike station name."},
        },
        "required": ["station_name"],
    },
)

trip_tool = Tool(function_declarations=[get_station_trip_count])

response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents="How many trips started at the Riverside station?",
    config=GenerateContentConfig(tools=[trip_tool]),
)

function_call = response.candidates[0].content.parts[0].function_call
print("Gemini requested a function call:")
print(" name:", function_call.name)
print(" args:", dict(function_call.args))
print("\n(Your code would now actually look this up and send the result back — that round trip")
print("is exactly what the ADK agent in the companion notebook automates for you.)")



### 4.5 Controlling Output With `GenerateContentConfig`
Every call so far used Gemini's defaults. This section asks the **exact same simple question** —
*"Write a one-sentence tagline for a new coffee shop."* — repeatedly, changing only one generation
parameter at a time, so the class can see cause and effect directly rather than take it on faith.

**The parameters, in plain terms:**
- **`temperature`** — how much randomness to inject when picking the next word. `0.0` is close to
  deterministic; higher values (up to `2.0`) let less likely words get chosen sometimes.
- **`max_output_tokens`** — a hard cap on response length. Once hit, the response stops **mid-word
  if necessary** — it's a length limit, not a "wrap up neatly" instruction.
- **`top_p`** (nucleus sampling) — only consider the smallest set of next-word candidates whose
  combined probability reaches `top_p`.
- **`top_k`** — only consider the `k` single most likely next words at each step, full stop.
- **`candidate_count`** — generate multiple independent completions for the *same* prompt in one call.
- **`stop_sequences`** — one or more strings that immediately halt generation the moment they're
  produced.

**Caveat for the class:** valid ranges can vary by model — check current docs for `GEMINI_MODEL_NAME`
if a value here gets rejected.


In [ ]:

DEMO_PROMPT = "Write a one-sentence tagline for a new coffee shop."

baseline = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=DEMO_PROMPT)
print("Baseline (no config passed, model defaults):")
print(" ", baseline.text)



#### 4.5.1 Temperature — Low vs. High
Run the *identical* prompt 3 times at each temperature. At `temperature=0.0`, expect the three
outputs to be identical or nearly so. At `temperature=1.8`, expect visibly different taglines each
time — same question, same model, only one number changed.


In [ ]:

from google.genai.types import GenerateContentConfig

print("=== temperature=0.0 (near-deterministic) ===")
for i in range(3):
    r = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=DEMO_PROMPT,
        config=GenerateContentConfig(temperature=0.0),
    )
    print(f" Run {i+1}:", r.text.strip())

print("\n=== temperature=1.8 (high creativity/randomness) ===")
for i in range(3):
    r = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=DEMO_PROMPT,
        config=GenerateContentConfig(temperature=1.8),
    )
    print(f" Run {i+1}:", r.text.strip())



#### 4.5.2 `max_output_tokens` — Hard Truncation
A deliberately tiny token budget (`max_output_tokens=8`) shows the response getting **cut off
mid-thought** — this is a length limit applied during generation, not a summarization instruction.


In [ ]:
r_full = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=DEMO_PROMPT,
    config=GenerateContentConfig(temperature=0.0, max_output_tokens=100),
)
print("max_output_tokens=100:", r_full.text.strip() if r_full.text else "(no text returned)")

r_short = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=DEMO_PROMPT,
    config=GenerateContentConfig(temperature=0.0, max_output_tokens=8),
)

if r_short.text:
    print("\nmax_output_tokens=8:", r_short.text.strip())
    print("(notice this one likely stops mid-sentence, not at a natural end point)")
else:
    finish_reason = r_short.candidates[0].finish_reason
    print(f"\nmax_output_tokens=8: (no visible text returned — finish_reason: {finish_reason})")
    print("This model reasons internally ('thinking') before producing visible text, and that")
    print("reasoning draws from the SAME max_output_tokens budget as the answer itself. At only 8")
    print("tokens, the entire budget was likely consumed by invisible reasoning, leaving nothing")
    print("for the actual response — an even more literal demonstration of a hard token cap than")
    print("mid-sentence truncation.")


#### 4.5.3 `top_p` and `top_k` — Two Different Ways to Narrow Word Choices
Both restrict which next-word candidates are eligible *before* temperature is even applied — `top_p`
by cumulative probability, `top_k` by a fixed count.


In [ ]:

print("=== top_p=0.1 (very narrow — only the most confident candidates) ===")
for i in range(2):
    r = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=DEMO_PROMPT,
        config=GenerateContentConfig(temperature=1.0, top_p=0.1),
    )
    print(f" Run {i+1}:", r.text.strip())

print("\n=== top_k=1 (only ever the single most likely next word — effectively greedy) ===")
for i in range(2):
    r = client.models.generate_content(
        model=GEMINI_MODEL_NAME,
        contents=DEMO_PROMPT,
        config=GenerateContentConfig(temperature=1.0, top_k=1),
    )
    print(f" Run {i+1}:", r.text.strip())



#### 4.5.4 `candidate_count` — Multiple Options From One Call
Instead of looping the API call yourself for variety, `candidate_count` asks for several independent
completions **in a single request**.


In [ ]:

r_multi = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=DEMO_PROMPT,
    config=GenerateContentConfig(temperature=1.0, candidate_count=3),
)

print(f"Got {len(r_multi.candidates)} candidates from one call:")
for i, candidate in enumerate(r_multi.candidates):
    print(f" Option {i+1}:", candidate.content.parts[0].text.strip())



#### 4.5.5 `stop_sequences` — Halting Generation on a Custom Marker
Ask for a numbered list, but set a stop sequence at `"3."` — generation halts the instant that
string would appear, cutting the list short regardless of `max_output_tokens`.


In [ ]:

r_stopped = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents="List 5 short tagline ideas for a coffee shop as a numbered list.",
    config=GenerateContentConfig(temperature=0.7, stop_sequences=["3."]),
)
print(r_stopped.text)
print("\n(Generation was cut off right before item 3, even though 5 items were requested.)")



### 4.6 Comparing Multiple Gemini Models Side by Side
**What this does:** the exact same prompt goes to three different models —
`gemini-2.5-flash-lite` (fastest, cheapest, lightest), `gemini-2.5-flash` (the balanced default used
everywhere else in this notebook), and `gemini-2.5-pro` (the most capable, slower, more expensive) —
and we compare **response quality, length, and latency** side by side. This is the practical
decision teams face in production: which model for which task.


In [ ]:

COMPARISON_PROMPT = (
    "A customer writes: 'My order arrived three days late and the box was crushed, "
    "but the product inside looks okay. I'm annoyed but I don't necessarily want a refund, "
    "just some acknowledgement.' Draft a short, empathetic customer service reply."
)

results = []
for model_name in COMPARISON_MODELS:
    start = time.time()
    r = client.models.generate_content(model=model_name, contents=COMPARISON_PROMPT)
    elapsed = time.time() - start
    results.append({
        "model": model_name,
        "latency_sec": round(elapsed, 2),
        "response_chars": len(r.text),
        "response": r.text.strip(),
    })

print(f"{'Model':<22} {'Latency (s)':<12} {'Length (chars)':<15}")
print("-" * 50)
for r in results:
    print(f"{r['model']:<22} {r['latency_sec']:<12} {r['response_chars']:<15}")

print("\nFull responses:\n")
for r in results:
    print(f"=== {r['model']} ===")
    print(r["response"])
    print()



**What to look for:** `gemini-2.5-flash-lite` typically responds fastest but sometimes more
generically; `gemini-2.5-pro` typically takes longer but often shows more nuance (e.g. correctly
picking up that the customer explicitly does *not* want a refund). Neither is "better" in the
abstract — the right choice depends on whether the task is high-volume/low-stakes (favor
flash-lite) or lower-volume/high-stakes (favor pro). This tradeoff is also where **provisioned
throughput** and batch inference (both under Agent Platform > Models > Deploy) come in for
production **scale** — pay-per-token works for bursty traffic, but a steady high-volume workload
often gets a better cost/latency profile from committing to provisioned throughput on the specific
model chosen here.

> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Garden** → search each model
> name used above — the model card shows context window size, supported modalities, and current
> pricing per model, which is what should drive this choice in a real project rather than guesswork.



### 4.7 Context Windows — How Much Text a Model Can Hold at Once
**What this does:** the "context window" is the total amount of text (measured in **tokens**, not
characters or words) a model can consider in a single call — your prompt, any conversation history,
and the model's own response all count against this same budget. We first check how many tokens a
tiny prompt uses, then build a much larger document and see how that scales, without needing a real
large dataset — the same short training-content sentences from Section 6 are simply repeated many
times to simulate "a lot of text."


In [ ]:

small_prompt = "Explain what a data lake is, in two sentences, for a business audience."

token_count = client.models.count_tokens(model=GEMINI_MODEL_NAME, contents=small_prompt)
print(f"Small prompt: {token_count.total_tokens} tokens")

# Simulate a much larger input by repeating a paragraph many times over — this is standing in for
# "a long document" (e.g. a full policy manual or transcript) without needing to source a real one.
paragraph = (
    "Cloud Storage stores objects in buckets with configurable storage classes and lifecycle rules. "
    "BigQuery partitioning splits a table by date or integer range to reduce bytes scanned per query. "
)
large_document = paragraph * 500  # roughly simulate a lengthy document

large_token_count = client.models.count_tokens(model=GEMINI_MODEL_NAME, contents=large_document)
print(f"Simulated large document: {large_token_count.total_tokens} tokens "
      f"(~{len(large_document):,} characters)")


In [ ]:

# Ask a question that can only be answered by reading the whole large_document, to confirm the
# model is actually attending to content late in a long context, not just the beginning.
needle = "The secret code word hidden in this document is PLATYPUS-77."
haystack_with_needle = large_document + " " + needle + " " + large_document

response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents=f"{haystack_with_needle}\n\nWhat is the secret code word mentioned in the text above?",
)
print("Model's answer:", response.text.strip())
print("\n(This is a simple 'needle in a haystack' test — if the model correctly returns")
print("PLATYPUS-77, it confirms it's actually reading content buried deep in a long context,")
print("not just skimming the start of the prompt.)")



**Why this matters at scale:** every token in the context window — input *and* output — is billed,
and very long contexts increase latency. This is the practical tradeoff behind **RAG** (Section 6):
instead of stuffing an entire knowledge base into every prompt, you retrieve only the few most
relevant chunks and include just those, keeping the context window small, fast, and cheap even as
your underlying document collection grows to millions of pages.

> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Garden** → open the model card
> for `gemini-2.5-flash` (or whichever model you're using) → the **context window** and **max output
> tokens** figures are listed directly on the card, so you can check them for any model without
> writing code.



### 4.8 Text Embeddings & Cosine Similarity
**What this demonstrates:** three sentences — two semantically similar, one unrelated — get
converted to embedding vectors, then compared pairwise using **cosine similarity** (a score from
-1 to 1 measuring how closely two vectors point in the same direction; for text embeddings, scores
in practice usually fall between 0 and 1, with higher meaning "more semantically similar"). Expect
the two similar sentences to score noticeably higher against each other than either does against
the unrelated third sentence — even though none of them share many exact words.


In [ ]:

import numpy as np

sentences = {
    "s1_cooking_a": "The chef prepared a delicious meal in the kitchen.",
    "s2_cooking_b": "A cook made a tasty dish in the kitchen.",
    "s3_unrelated": "The stock market fell sharply today.",
}

embed_response = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=list(sentences.values()),
)
vectors = {key: np.array(e.values) for key, e in zip(sentences.keys(), embed_response.embeddings)}

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Sentences:")
for key, text in sentences.items():
    print(f"  {key}: {text}")

print("\nPairwise cosine similarity:")
keys = list(sentences.keys())
for i in range(len(keys)):
    for j in range(i + 1, len(keys)):
        sim = cosine_similarity(vectors[keys[i]], vectors[keys[j]])
        print(f"  {keys[i]} <-> {keys[j]}: {sim:.4f}")

print("\nExpect s1_cooking_a <-> s2_cooking_b to score clearly highest — same meaning, different words.")



### 4.9 Task-Based Embeddings
Embedding models accept an optional `task_type` that changes **how the same text is embedded**,
optimized for what you intend to do with it. Common values: `RETRIEVAL_QUERY` (for a search query),
`RETRIEVAL_DOCUMENT` (for the documents being searched), `SEMANTIC_SIMILARITY`, `CLASSIFICATION`,
and `CLUSTERING`.

#### 4.9.1 Same Text, Different Task Types — Different Vectors


In [ ]:

from google.genai.types import EmbedContentConfig

same_text = "How do I bake a chocolate cake?"

emb_as_query = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=same_text,
    config=EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
).embeddings[0].values

emb_as_document = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=same_text,
    config=EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
).embeddings[0].values

emb_as_similarity = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=same_text,
    config=EmbedContentConfig(task_type="SEMANTIC_SIMILARITY"),
).embeddings[0].values

v_query = np.array(emb_as_query)
v_doc = np.array(emb_as_document)
v_sim = np.array(emb_as_similarity)

print("Same sentence, three task_types — cosine similarity between the resulting vectors:")
print(f"  RETRIEVAL_QUERY <-> RETRIEVAL_DOCUMENT:   {cosine_similarity(v_query, v_doc):.4f}")
print(f"  RETRIEVAL_QUERY <-> SEMANTIC_SIMILARITY:  {cosine_similarity(v_query, v_sim):.4f}")
print(f"  RETRIEVAL_DOCUMENT <-> SEMANTIC_SIMILARITY: {cosine_similarity(v_doc, v_sim):.4f}")
print("\nNone of these should be exactly 1.0 — proof task_type changes the actual embedding,")
print("not just a label attached to an otherwise identical vector.")



#### 4.9.2 The Practical Payoff: Query vs. Document Task Types in Retrieval
This is *why* task types exist. A query is embedded as `RETRIEVAL_QUERY`; two candidate documents
— one relevant, one not — are embedded as `RETRIEVAL_DOCUMENT`. Expect the relevant document to
score clearly higher against the query — this asymmetric query/document embedding is exactly what
Section 6's Vector Search index relies on for RAG.


In [ ]:

query_text = "How do I bake a chocolate cake?"
relevant_doc = "This recipe explains step-by-step how to bake a delicious chocolate cake at home."
irrelevant_doc = "The history of the Roman Empire spans over a thousand years."

query_vec = np.array(
    client.models.embed_content(
        model=EMBEDDING_MODEL_NAME,
        contents=query_text,
        config=EmbedContentConfig(task_type="RETRIEVAL_QUERY"),
    ).embeddings[0].values
)

doc_vecs = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=[relevant_doc, irrelevant_doc],
    config=EmbedContentConfig(task_type="RETRIEVAL_DOCUMENT"),
).embeddings

relevant_vec = np.array(doc_vecs[0].values)
irrelevant_vec = np.array(doc_vecs[1].values)

print("Query:", query_text)
print(f"\nSimilarity to RELEVANT doc   ('{relevant_doc[:40]}...'): {cosine_similarity(query_vec, relevant_vec):.4f}")
print(f"Similarity to IRRELEVANT doc ('{irrelevant_doc[:40]}...'): {cosine_similarity(query_vec, irrelevant_vec):.4f}")
print("\nExpect the relevant document to score clearly higher — this is retrieval working correctly.")



## 5. Grounding With Google Search
**What this does:** rather than answering purely from training data, the `GoogleSearch` tool lets
Gemini issue real Google Search queries and ground its answer in current results — the fastest,
cheapest way to demo grounding, with no infrastructure to deploy (contrast with Section 6, where we
ground against *our own* documents instead of the public web).


In [ ]:

search_tool = Tool(google_search=GoogleSearch())

response = client.models.generate_content(
    model=GEMINI_MODEL_NAME,
    contents="What GCP announcements happened in the last week?",
    config=GenerateContentConfig(tools=[search_tool]),
)
print(response.text)

if response.candidates[0].grounding_metadata and response.candidates[0].grounding_metadata.grounding_chunks:
    print("\nGrounding sources used:")
    for chunk in response.candidates[0].grounding_metadata.grounding_chunks:
        if chunk.web:
            print(" -", chunk.web.title, "|", chunk.web.uri)



> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Garden** → open a Gemini model
> card → the **Grounding** section of the docs linked there lists Google Search grounding pricing,
> which is billed per grounded request, separate from the underlying token cost.



## 6. Vector Search — Grounding on Your Own Documents (RAG)
**Note:** Vector Search itself (`google.cloud.aiplatform.MatchingEngine*`) is unaffected by the SDK
deprecation above — only the embedding *generation* call below has moved to `google-genai`.

### 6.1 Embed a Handful of Small Documents
**What this does:** converts each short text document into a numeric embedding vector — documents
with similar meaning end up with similar vectors, which is what makes semantic search possible.
We use only 5 tiny documents to keep index build time and cost minimal.


In [ ]:

documents = [
    "Cloud Storage stores objects in buckets with configurable storage classes and lifecycle rules.",
    "BigQuery partitioning splits a table by date or integer range to reduce bytes scanned per query.",
    "Pub/Sub delivers messages to every subscription attached to a topic — this is the fan-out pattern.",
    "Vertex AI Feature Store centralizes features for consistent use across training and serving.",
    "Dataflow runs Apache Beam pipelines for both batch and streaming data processing on GCP.",
]

embed_response = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=documents,
)
embeddings = [e.values for e in embed_response.embeddings]

print(f"Generated {len(embeddings)} embeddings, each with {len(embeddings[0])} dimensions.")



### 6.2 Build & Deploy a Small Vector Search Index
**Cost/time note:** this is the one resource in this notebook that bills per node-hour once
deployed. `e2-standard-2` (the smallest supported endpoint machine type) and a tiny 5-document
index keep both build time (a few minutes) and ongoing cost as low as possible. **Undeploy and
delete this right after the demo (Section 6.4) — don't leave it running.**

**On scale:** this same `MatchingEngineIndex` API is what backs Vector Search indexes with
*billions* of vectors in production — the only things that change moving from this 5-document demo
to a real large-scale index are `approximate_neighbors_count`, `shard_size`, and the machine
type/replica count on the deployed endpoint, not the underlying workflow.


In [ ]:

import json

os.makedirs("vector_search_data", exist_ok=True)
with open("vector_search_data/embeddings.json", "w") as f:
    for i, emb in enumerate(embeddings):
        f.write(json.dumps({"id": str(i), "embedding": emb}) + "\n")

EMBEDDINGS_BUCKET = f"gs://{PROJECT_ID}-vector-search-{_suffix}"
!gsutil mb -l {REGION} {EMBEDDINGS_BUCKET}
!gsutil cp vector_search_data/embeddings.json {EMBEDDINGS_BUCKET}/embeddings.json


In [ ]:

from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=REGION)

index = aiplatform.MatchingEngineIndex.create_tree_ah_index(
    display_name=f"training-doc-index-{_suffix}",
    contents_delta_uri=EMBEDDINGS_BUCKET,
    dimensions=len(embeddings[0]),
    approximate_neighbors_count=5,
    distance_measure_type="DOT_PRODUCT_DISTANCE",
    index_update_method="BATCH_UPDATE",
    # Must be explicit: the default shard size (SHARD_SIZE_MEDIUM) is incompatible with the
    # smallest deployment machine type (e2-standard-2) used below — SMALL is required to pair
    # with the cheapest possible endpoint machine type.
    shard_size="SHARD_SIZE_SMALL",
    # Explicit small-corpus-safe values — defaults are tuned for much larger indexes and can
    # behave oddly (e.g. empty results) on a toy 5-document index like this one.
    leaf_node_embedding_count=5,
    leaf_nodes_to_search_percent=100,
)
print("Created index:", index.resource_name)

index_endpoint = aiplatform.MatchingEngineIndexEndpoint.create(
    display_name=f"training-doc-endpoint-{_suffix}",
    public_endpoint_enabled=True,
)

DEPLOYED_INDEX_ID = f"deployed_{_suffix}"
index_endpoint.deploy_index(
    index=index,
    deployed_index_id=DEPLOYED_INDEX_ID,
    machine_type="e2-standard-2",
    min_replica_count=1,
    max_replica_count=1,
)
print("Deployed index endpoint:", index_endpoint.resource_name)



> 🖥️ **Check it in the console:** **Agent Platform > Models > Vector Search** → **Indexes** shows
> `training-doc-index-...` building/ready, and **Index Endpoints** shows the deployed endpoint with
> its machine type and node count.

### 6.3 Query the Index & Ground a Gemini Answer
**What this does:** embed the *question* with the same embedding model, find the most similar
stored document via Vector Search, then inject that document's text directly into the Gemini
prompt as context — the core pattern behind Retrieval-Augmented Generation (RAG).


In [ ]:
import time
from google.api_core.exceptions import ServiceUnavailable

# Re-fetch a fresh index_endpoint object rather than reusing the one that's been sitting idle
# through the whole deployment wait -- its gRPC channel may have gone stale.
index_endpoint = aiplatform.MatchingEngineIndexEndpoint(index_endpoint.resource_name)

query_text = "How does BigQuery reduce the amount of data scanned per query?"
query_embedding = client.models.embed_content(
    model=EMBEDDING_MODEL_NAME,
    contents=[query_text],
).embeddings[0].values

for attempt in range(5):
    try:
        neighbors = index_endpoint.find_neighbors(
            deployed_index_id=DEPLOYED_INDEX_ID,
            queries=[query_embedding],
            num_neighbors=2,
        )
        break
    except ServiceUnavailable as e:
        wait = 10 * (attempt + 1)
        print(f"attempt {attempt + 1} hit a transient connection error, retrying in {wait}s...")
        time.sleep(wait)
else:
    raise RuntimeError("find_neighbors failed after 5 attempts — see troubleshooting note below")

retrieved_docs = [documents[int(n.id)] for n in neighbors[0]]
print("Retrieved context:", retrieved_docs)

grounded_prompt = f'''Answer the question using ONLY the context below.

Context:
{chr(10).join(retrieved_docs)}

Question: {query_text}'''

rag_response = client.models.generate_content(model=GEMINI_MODEL_NAME, contents=grounded_prompt)
print("\nGrounded answer:\n", rag_response.text)


### 6.4 Undeploy & Delete the Index — Do This Now
This is the billed resource from this section — clean it up as soon as the demo is done.


In [ ]:

index_endpoint.undeploy_index(deployed_index_id=DEPLOYED_INDEX_ID)
index_endpoint.delete(force=True)
index.delete()
!gsutil -m rm -r {EMBEDDINGS_BUCKET}
print("Vector Search index, endpoint, and staging bucket deleted.")



> 🖥️ **Check it in the console:** **Agent Platform > Models > Vector Search** → both the index and
> endpoint from this section should now show as deleted / no longer listed. If either still appears,
> delete it manually from there before moving on.



## 7. Model Garden (Guided Walkthrough)
**Why this is a walkthrough, not a live exercise:** Model Garden is primarily a browsing/deployment
UI in the console — deploying any of its open models (Llama, Gemma, etc.) creates a dedicated
endpoint that bills per node-hour, same as Section 6's Vector Search endpoint, but typically on a
larger and more expensive machine type. Walk through the console instead of deploying live:

- Model Garden catalogs **200+ models**: first-party (Gemini), open (Gemma, Llama), and third-party
  partner models (including Anthropic's Claude Opus, Sonnet, and Haiku, available as partner models
  in supported regions).
- Deployment options vary by model: some deploy to a dedicated endpoint (node-hour billing), others
  are available as pay-per-token Model-as-a-Service with zero idle cost — worth pointing out to the
  class as the key cost-model distinction to check before deploying anything.
- **On scale:** for a steady, high-volume workload on a specific model, **Provisioned Throughput**
  (under Agent Platform > Models > Deploy) reserves guaranteed capacity at a flatter, more
  predictable cost than pay-per-token — this is the production answer to "what happens when demand
  grows," in contrast to the ad-hoc pay-per-call pattern used everywhere else in this notebook.
- If you do want a live moment here, an open model on the smallest supported machine type is the
  lowest-cost open-model deployment option — but confirm current pricing/model availability before
  committing to it live.

> 🖥️ **Check it in the console:** **Agent Platform > Models > Model Garden** → filter by "Partner
> models" to see Claude alongside Gemini, or search "Gemma" to see an open model's deployment
> options and machine-type requirements directly on its card.


In [ ]:
!gcloud ai model-garden models list --limit=10


## 8. Agent Studio, formerly Agent Builder (Guided Walkthrough)
**Naming note:** what used to be called "Vertex AI Agent Builder" is now **Agent Studio**, one of
several tools under the **Build** pillar of Agent Platform's separate **Agents** section (distinct
from the **Models** section this whole notebook has lived under).

**Why this is a walkthrough here, not a live exercise:** Agent Studio is a low-code visual canvas —
console-driven by design. For the code-first, hands-on equivalent, see the companion notebook
**Agentic AI on Gemini Enterprise Agent Platform**, which builds the same underlying concept using
the Agent Development Kit (ADK) instead.

- **The core idea:** an agent is a loop around exactly the function-calling mechanism from
  Section 4.4 — the model decides which tool to call and with what arguments, your code (or Agent
  Studio's managed runtime) executes it, the result goes back to the model, and this repeats until
  the model produces a final answer instead of another tool call.
- **What Agent Studio adds on top:** a managed orchestration loop (so you don't hand-write the
  "call tool → feed result back → repeat" loop yourself), grounding against your own data stores,
  and a conversational UI/widget you can embed — all with less custom code than wiring the loop up
  by hand.
- **Cost model:** billed per conversation/query, not per node-hour — closer to the Gemini API's
  cost profile than to a deployed endpoint.

> 🖥️ **Check it in the console:** **Agent Platform > Agents > Build > Agent Studio** → click
> **Create** to see the visual canvas for defining an agent's goal, tools, and data stores, without
> writing any code. Compare this with **Agent Platform > Agents > Build > Agent Garden**, a library
> of prebuilt agent templates for common patterns — worth a look before building anything from
> scratch, live or in the companion notebook.



## 9. Cleanup & Final Verification
Section 6.4 already deletes the one resource in this notebook that bills continuously (the Vector
Search index endpoint) — this section is a final safety-net check, not a second cleanup pass.
Sections 7 and 8 are guided walkthroughs only (read-only `gcloud` list command and no code
execution at all) — nothing to delete there **unless you personally deployed a Model Garden model
live**, in which case that endpoint is not tracked by this notebook and needs manual deletion.


In [ ]:

print("Checking for any lingering Vector Search indexes/endpoints from this notebook...")
!gcloud ai indexes list --region={REGION} --filter="displayName:training-doc-index" --format="value(name)"
!gcloud ai index-endpoints list --region={REGION} --filter="displayName:training-doc-endpoint" --format="value(name)"

print("\nBoth commands above should return nothing if Section 6.4 ran successfully.")
print("If either lists something, delete it with:")
print("  gcloud ai index-endpoints delete ENDPOINT_ID --region={REGION} --quiet   (undeploy first if needed)")
print("  gcloud ai indexes delete INDEX_ID --region={REGION} --quiet")

print("\nChecking for any lingering buckets created by this notebook...")
!gsutil ls -p {PROJECT_ID} | grep -i "vector-search" || echo "None found."

print("\nReminder: if you deployed a real Model Garden model in Section 7 (outside this notebook's")
print("tracked variables), check Agent Platform > Models > Deploy > Online prediction endpoints in")
print("the console and delete it manually — that resource isn't created or tracked by any cell here.")



> 🖥️ **Final console check:** **Agent Platform > Models > Vector Search** (both Indexes and Index
> Endpoints tabs) and **Agent Platform > Models > Deploy > Online prediction** should show nothing
> from this notebook remaining. If you're moving on to the companion Agentic AI notebook next, no
> further cleanup is needed here first — the two notebooks use independent resources.
